# 11 — The moments a joint state reads

`04/10` turned signals into states: three per-sample embeddings, a covariance density for a whole record, and the time-averaged joint state $\rho_{AB}$ of two signals, whose every entry is the time average of one phase factor. It left two bright entries in the joint state of a locked pair without saying what they hold. This notebook reads them. You will find which **circular moments of the phase difference** each embedding's joint state carries, and see on a small toy what an embedding that carries one more moment can tell apart.

**Prerequisites:** `04/10_embedding_signals_into_states` (the embeddings and the joint state), `02/11_quantum_mutual_information` (the quantum mutual information of a bipartite state).

**What you'll be able to do**
- Show by hand that, when signal $A$ sweeps the circle and the phase difference $\delta = \theta_A - \theta_B$ is independent of it, the only entries of $\rho_{AB}$ that survive are the circular moments $\langle e^{im\delta}\rangle$ of the difference — and break that rule on purpose to see what leaks in otherwise.
- Locate the moments: the phase qubit's coherence $\lvert\rho[1,2]\rvert$ is exactly a quarter of the phase-locking value (PLV), and the multi-frequency qutrit adds $\lvert\rho[6,2]\rvert = \lvert\langle e^{2i\delta}\rangle\rvert/9$, the second-harmonic PLV divided by nine.
- Build two phase-difference signals with the **same first circular moment** but **different second moment**, and show that PLV and the naive embedding cannot tell them apart while the qutrit's second-moment coherence, and the classical second-harmonic PLV beside it, can.
- Explain why the quantum mutual information of the qutrit state differs between the two signals yet is **not** a gauge of the second moment: at a fixed first moment it is not monotone, and for a symmetric difference law the second moment cannot fall below twice the square of the first, minus one.

Today's question is the hinge of the capstone, kept small enough to hold in your head: which numbers about the coupling can a joint state contain at all? Once you can answer it for any embedding, you can predict what a quantum measure of that state is able to see before you compute it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS, CMAP_PLAN
from qot_course.quantum.composite import partial_trace
from qot_course.quantum_ot.capstone import joint_density_matrix, plv
from qot_course.quantum_ot.embeddings import multifreq_state, joint_density_from_states
from qot_course.quantum_ot.biharmonic import long_run_joint_density
from qot_course.infotheory.quantum import quantum_mutual_information

# The von Mises draws of section 1 come from this generator; the two-signal toy of
# section 2 seeds its own shuffle generator, so its numbers do not depend on section 1.
rng = np.random.default_rng(7)
viz.use_course_style()

## 1. Which entries survive, and what they hold

Start from the entry formula of `04/10`. For the phase qubit, $\rho_{AB}[(ij),(kl)] = \tfrac14\langle e^{\mathrm{i}(m\theta_A + n\theta_B)}\rangle$ with $m = i - k$ and $n = j - l$, where the upright $\mathrm{i}$ is the imaginary unit and $i$ a basis index. For the multi-frequency qutrit the same computation gives $\tfrac19\langle e^{\mathrm{i}(m\theta_A + n\theta_B)}\rangle$, now with $m, n \in \{-2, \dots, 2\}$, at flat index $3i + j$.

**The phase difference enters.** Write $\theta_B = \theta_A - \delta$. Then $m\theta_A + n\theta_B = (m + n)\theta_A - n\delta$. Two conditions make the bookkeeping clean: signal $A$ sweeps the circle uniformly (a *phase-symmetric reference*), and $\delta$ is independent of $\theta_A$. Under both, the average factorises into $\langle e^{i(m+n)\theta_A}\rangle\,\langle e^{-in\delta}\rangle$, and the first factor vanishes unless $m + n = 0$. **Only the entries with $n = -m$ survive, and each holds $\langle e^{im\delta}\rangle$, the $m$-th circular moment of the difference** (Mardia & Jupp 2000). Every other off-diagonal entry, the marginal coherences included, averages to zero.

**Where the moments sit.**

- *Phase qubit.* The entry linking $\lvert 01\rangle$ (row, flat index $1$) and $\lvert 10\rangle$ (column, flat index $2$) has $m = -1$, $n = 1$, so $\rho[1,2] = \tfrac14\langle e^{-i\delta}\rangle$. The **phase-locking value** is the modulus of the first moment, $\mathrm{PLV} = \lvert\langle e^{i\delta}\rangle\rvert$ (Lachaux et al. 1999), hence $\lvert\rho[1,2]\rvert = \mathrm{PLV}/4$ exactly. These were the two bright entries of `04/10`. **The naive embedding is PLV-equivalent by construction.**
- *Multi-frequency qutrit.* The first moment appears in several entries, among them $\rho[3,1] = \tfrac19\langle e^{i\delta}\rangle$ (row $\lvert 10\rangle$, column $\lvert 01\rangle$). The second moment appears in one pair only: $\rho[6,2] = \tfrac19\langle e^{2i\delta}\rangle$ (row $\lvert 20\rangle$, column $\lvert 02\rangle$), a place the qubit does not even have.

Read that last line with a classical eye. $\lvert\langle e^{2i\delta}\rangle\rvert$ is itself a classical statistic, the **second-harmonic PLV**. The qutrit's extra entry is that statistic divided by nine. Under these two conditions, a joint state built this way holds a list of classical moments of the difference and nothing else, up to the residue of a finite record, so any measure computed from it is a function of those moments. That is the honest frame for everything the capstone measures.

The code checks the three identities on a phase difference drawn from a von Mises law, the circle's analogue of the Gaussian, here centred at $0.5$ rad with concentration $\kappa = 2$ (Mardia & Jupp 2000), then breaks the independence condition on purpose: the same values of $\delta$, laid out in rising order along the sweep of $\theta_A$.

In [ ]:
n = 60_000
theta_a = np.linspace(0.0, 2 * np.pi, n, endpoint=False)    # sweeps the circle uniformly
delta = rng.vonmises(0.5, 2.0, n)        # iid draws: independent of theta_A by construction
theta_b = theta_a - delta

m1 = np.mean(np.exp(1j * delta))
m2 = np.mean(np.exp(2j * delta))
rho_q = joint_density_matrix(theta_a, theta_b)
rho_t = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b))
print(f"first moment  <e^(i delta)>  = {m1:.4f}   PLV = {abs(m1):.4f}")
print(f"second moment <e^(2i delta)> = {m2:.4f}   second-harmonic PLV = {abs(m2):.4f}")
print(f"qubit   rho[1,2] = {rho_q[1, 2]:.4f}    conj(m1)/4 = {np.conj(m1) / 4:.4f}    4|rho[1,2]| = {4 * abs(rho_q[1, 2]):.4f}")
print(f"qutrit  rho[3,1] = {rho_t[3, 1]:.4f}    m1/9 = {m1 / 9:.4f}")
print(f"qutrit  rho[6,2] = {rho_t[6, 2]:.4f}    m2/9 = {m2 / 9:.4f}    9|rho[6,2]| = {9 * abs(rho_t[6, 2]):.4f}")


def survivors(rho, d=3):
    '''Largest off-diagonal |entry| with n = -m, by |m|, and largest with n != -m.'''
    by_m, others = {}, 0.0
    for r in range(d * d):
        for c in range(d * d):
            if r == c:
                continue
            m_, n_ = r // d - c // d, r % d - c % d
            if n_ == -m_:
                by_m[abs(m_)] = max(by_m.get(abs(m_), 0.0), abs(rho[r, c]))
            else:
                others = max(others, abs(rho[r, c]))
    return by_m, others


by_m, others = survivors(rho_t)
print()
print(f"qutrit entries with n = -m: largest at |m| = 1: {by_m[1]:.4f}, at |m| = 2: {by_m[2]:.4f};  "
      f"largest entry with n != -m: {others:.4f}")

# Break the independence condition: same values of delta, in rising order along theta_A.
theta_b_sorted = theta_a - np.sort(delta)
rho_t_sorted = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b_sorted))
print()
for name, rho in [("independent order", rho_t), ("sorted order", rho_t_sorted)]:
    rho_b = partial_trace(rho, keep=[1], dims=[3, 3])
    marg = np.abs(rho_b - np.diag(np.diag(rho_b))).max()
    print(f"{name:<18} |rho[6,2]| = {abs(rho[6, 2]):.4f}   largest off-diagonal of rho_B = {marg:.4f}   "
          f"largest entry with n != -m = {survivors(rho)[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"wspace": 0.25})
ticks = [f"{i}{j}" for i in range(3) for j in range(3)]
for ax, rho, title in [(axes[0], rho_t, "Independent order: moments only"),
                       (axes[1], rho_t_sorted, "Sorted order: other entries leak in")]:
    im = ax.imshow(np.abs(rho), cmap=CMAP_PLAN, vmin=0.0, vmax=1 / 9)
    ax.set_xticks(range(9))
    ax.set_xticklabels(ticks, fontsize=8)
    ax.set_yticks(range(9))
    ax.set_yticklabels(ticks, fontsize=8)
    ax.set_xlabel(r"column $|k\,l\rangle$")
    ax.set_ylabel(r"row $|i\,j\rangle$")
    ax.set_title(title, pad=10)
    ax.grid(False)
for r, c in [(6, 2), (2, 6)]:
    axes[0].add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec=COLORS["highlight"], lw=2.2))
fig.colorbar(im, ax=axes, shrink=0.8, label=r"$|\rho_{AB}|$ entry (qutrit embedding)")
plt.show()

**Read the output and the figure.** The three identities hold. The printed $4\lvert\rho[1,2]\rvert$ equals the PLV, $\rho[3,1]$ equals $m_1/9$, and $9\lvert\rho[6,2]\rvert$ equals the second-harmonic PLV, to every printed digit. They are exact here, not approximate: for an entry with $n = -m$ the factor in $\theta_A$ is $e^{0} = 1$ for every sample, so the entry *is* the sample moment divided by four or nine.

The second block lists the survivors. The entries with $n = -m$ carry the first moment (largest modulus $0.0776$, at $\lvert m\rvert = 1$) and the second ($0.0336$, at $\lvert m\rvert = 2$); the largest entry of any other kind is $0.0007$, about fifty times smaller than the smaller of the two, the residue of a finite record. *Left:* the qutrit's joint state, with the diagonal at $\tfrac19$, a scatter of first-moment entries, and the rose squares on the second-moment pair $\rho[6,2]$ and $\rho[2,6]$. Everything else is dark.

*Right:* the same values of $\delta$, sorted. The moment entries do not move, since a sample moment does not depend on the order of the samples, and the third block prints the same $\lvert\rho[6,2]\rvert$ for both layouts. But now $\delta$ rises with $\theta_A$, so $\theta_B$ on its own prefers some phases, and entries that should vanish light up: the largest of them reaches $0.0982$, and the largest coherence of the marginal $\rho_B$ goes from $0.0014$ to $0.1842$. That preference says nothing about the coupling, and every measure computed from the state would pick it up. This is why the toy of section 2 shuffles its phase differences before use: the independence condition is part of the construction, not a detail.

## 2. Same first moment, different second moment

Now the contrast that motivates the whole capstone, kept deliberately small. We build **two** phase-difference signals engineered to be indistinguishable to PLV yet distinguishable by the qutrit, and watch each embedding react.

If two signals share the **same** first moment but have **different** second moments, section 1 says the naive coherence (and PLV) must match, while the qutrit's second-moment coherence must differ. We construct them with a transparent two-population recipe. Let $\theta_A$ sweep the circle uniformly. Then assign the per-sample difference $\delta$ to fixed-fraction populations, in an order shuffled once so that $\delta$ is independent of $\theta_A$:

- **Signal 1** — an equal split between $\delta = +s_1$ and $\delta = -s_1$. This is symmetric, so its first moment is real and equals $\cos s_1$; its second moment is $\cos 2s_1$.
- **Signal 2** — a fraction $w$ of samples at $\delta = 0$ mixed with an equal split between $\delta = +s_2$ and $\delta = -s_2$. Its first moment is $w + (1-w)\cos s_2$ and its second is $w + (1-w)\cos 2s_2$.

By hand, with $s_1 = 0.6$: the first moment is $\cos 0.6 \approx 0.825$ and the second $\cos 1.2 \approx 0.362$. With $s_2 = 1.2$, matching the first moment asks $w + (1 - w)\cos 1.2 = \cos 0.6$, so $w = (\cos 0.6 - \cos 1.2)/(1 - \cos 1.2) \approx 0.726$, and then the second moment is $w + (1 - w)\cos 2.4 \approx 0.524$. The code solves the same equations and builds the phases.

In [ ]:
# Two phase-difference signals: matched FIRST circular moment, distinct SECOND.
# theta_A sweeps the circle uniformly and the difference is drawn independently of it
# -> a phase-symmetric reference, so only the DIFFERENCE moments survive in the joint
# coherences.
n = 180_000
theta_a = np.linspace(0.0, 2 * np.pi, n, endpoint=False)


def circular_moments(weight_zero, spread):
    """First and second circular moments of the two-population difference recipe."""
    first = weight_zero + (1.0 - weight_zero) * np.cos(spread)
    second = weight_zero + (1.0 - weight_zero) * np.cos(2.0 * spread)
    return first, second


# Signal 1: pure symmetric two-point split at +/- s1 (no spike).
s1 = 0.6
first_target, second_1 = circular_moments(0.0, s1)

# Signal 2: choose s2, then solve the spike weight w2 so the FIRST moment matches.
s2 = 1.2
w2 = (first_target - np.cos(s2)) / (1.0 - np.cos(s2))
first_2, second_2 = circular_moments(w2, s2)

print('design (analytic targets):')
print(f'  signal 1:  first moment = {first_target:.4f}   second moment = {second_1:.4f}')
print(f'  signal 2:  first moment = {first_2:.4f}   second moment = {second_2:.4f}   (spike weight w = {w2:.4f})')
print(f'  first moments matched: {bool(np.isclose(first_target, first_2))}    second moments distinct: {bool(not np.isclose(second_1, second_2))}')


def difference_signal(weight_zero, spread, n_total, rng):
    """Per-sample phase difference: exact fixed-fraction populations, in shuffled order."""
    delta = np.empty(n_total)
    n_zero = int(round(weight_zero * n_total))
    delta[:n_zero] = 0.0
    n_rest = n_total - n_zero
    n_plus = n_rest // 2
    delta[n_zero:n_zero + n_plus] = spread
    delta[n_zero + n_plus:] = -spread
    # Shuffle so that delta is independent of theta_A (the condition of section 1).
    rng.shuffle(delta)
    return delta


shuffle_rng = np.random.default_rng(42)
delta_1 = difference_signal(0.0, s1, n, shuffle_rng)
delta_2 = difference_signal(w2, s2, n, shuffle_rng)
theta_b1 = theta_a - delta_1
theta_b2 = theta_a - delta_2

In [ ]:
# Read every quantity off both signals.
# Classical baselines: PLV (first moment) and the second-harmonic PLV (second moment).
plv_1 = plv(theta_a, theta_b1)
plv_2 = plv(theta_a, theta_b2)
plv2_1 = abs(np.mean(np.exp(2j * (theta_a - theta_b1))))
plv2_2 = abs(np.mean(np.exp(2j * (theta_a - theta_b2))))

# Naive phase-qubit joint density: the PLV-level coherence is rho[1, 2] (= <e^{-i delta}>/4).
rho_naive_1 = joint_density_matrix(theta_a, theta_b1)
rho_naive_2 = joint_density_matrix(theta_a, theta_b2)
naive_coh_1 = abs(rho_naive_1[1, 2])
naive_coh_2 = abs(rho_naive_2[1, 2])

# Multi-frequency qutrit joint density: first moment at rho[3, 1], second at rho[6, 2].
rho_mf_1 = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b1))
rho_mf_2 = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b2))
mf_first_1, mf_second_1 = abs(rho_mf_1[3, 1]), abs(rho_mf_1[6, 2])
mf_first_2, mf_second_2 = abs(rho_mf_2[3, 1]), abs(rho_mf_2[6, 2])

header = f"{'quantity':<46}{'signal 1':>12}{'signal 2':>12}"
print(header)
print('-' * len(header))
print(f"{'PLV  |<e^{i delta}>|  (classical)':<46}{plv_1:>12.4f}{plv_2:>12.4f}")
print(f"{'naive coherence  |rho[1,2]|  (= PLV/4)':<46}{naive_coh_1:>12.4f}{naive_coh_2:>12.4f}")
print(f"{'multifreq 1st moment  |rho[3,1]|':<46}{mf_first_1:>12.4f}{mf_first_2:>12.4f}")
print(f"{'2nd-harmonic PLV  |<e^{2i delta}>|  (classical)':<46}{plv2_1:>12.4f}{plv2_2:>12.4f}")
print(f"{'multifreq 2nd moment  |rho[6,2]|':<46}{mf_second_1:>12.4f}{mf_second_2:>12.4f}")

# Section 1's warning, checked: the shuffle keeps the marginals free of coherence.
for name, rho in [("signal 1", rho_mf_1), ("signal 2", rho_mf_2)]:
    rho_b = partial_trace(rho, keep=[1], dims=[3, 3])
    print(f"{name}: largest off-diagonal of the qutrit marginal rho_B = "
          f"{np.abs(rho_b - np.diag(np.diag(rho_b))).max():.4f}")

# A coupling readout on each joint state: the quantum mutual information.
qmi_naive_1 = quantum_mutual_information(rho_naive_1, dims=[2, 2], base=np.e)
qmi_naive_2 = quantum_mutual_information(rho_naive_2, dims=[2, 2], base=np.e)
qmi_1 = quantum_mutual_information(rho_mf_1, dims=[3, 3], base=np.e)
qmi_2 = quantum_mutual_information(rho_mf_2, dims=[3, 3], base=np.e)
print()
print(f'naive QMI (nats):            signal 1 = {qmi_naive_1:.4f}   signal 2 = {qmi_naive_2:.4f}')
print(f'multi-frequency QMI (nats):  signal 1 = {qmi_1:.4f}   signal 2 = {qmi_2:.4f}   '
      f'relative gap = {(qmi_1 - qmi_2) / qmi_1:.1%}')

**Read the output.** Read the table top to bottom. The first two rows — **PLV** and the **naive coherence** $\lvert\rho[1,2]\rvert$ — print the *same* value for both signals, $0.8253$ and $0.2063$, and the second is exactly a quarter of the first. By construction the two signals share their first circular moment, and the naive embedding sees *only* the first moment, so to PLV and to the phase qubit these two signals are **identical**. A discriminating analysis built on either one cannot tell them apart. The third row confirms that the qutrit's *first*-moment coherence $\lvert\rho[3,1]\rvert$ also matches, at $0.0917$, as it must.

The decisive rows are the last two. The qutrit's *second*-moment coherence $\lvert\rho[6,2]\rvert$ is **different** for the two signals, $0.0403$ against $0.0582$, and nine times each value is the second-harmonic PLV printed directly above it, $0.3624$ against $0.5241$. The qutrit state genuinely differs between signal 1 and signal 2, in a place the qubit does not have, and the classical statistic that sits beside it differs in exactly the same proportion, since the entry is that statistic divided by nine. The two marginal checks print values near zero: the shuffle did its job, and no one-signal preference leaks into these readings.

The quantum mutual information agrees with both halves of that reading. On the naive joint state it prints the same value for the two signals, $0.1984$ nat: the phase qubit has nothing to tell them apart with. On the qutrit joint state it prints $0.3701$ nat for signal 1 and $0.3629$ nat for signal 2: two different states, two different values. Section 3 looks hard at that last gap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"wspace": 0.25})

# (a) The two difference distributions: same first moment, different shape.
ax = axes[0]
bins = np.linspace(-np.pi, np.pi, 60)
ax.hist(np.mod(delta_1 + np.pi, 2 * np.pi) - np.pi, bins=bins, density=True,
        color=COLORS["quantum"], alpha=0.7, label="signal 1")
ax.hist(np.mod(delta_2 + np.pi, 2 * np.pi) - np.pi, bins=bins, density=True,
        color=COLORS["highlight"], alpha=0.6, label="signal 2")
ax.set_xlabel(r"phase difference  $\delta = \theta_A - \theta_B$  (radians)")
ax.set_ylabel("density")
ax.set_title("Two difference signals, matched first moment", pad=10)
ax.legend()

# (b) Every reading on the scale of a moment: each entry times its prefactor.
ax = axes[1]
groups = ["PLV", "4 |rho[1,2]|\nnaive", "9 |rho[3,1]|\nqutrit", "2nd-harm.\nPLV", "9 |rho[6,2]|\nqutrit"]
vals_1 = [plv_1, 4 * naive_coh_1, 9 * mf_first_1, plv2_1, 9 * mf_second_1]
vals_2 = [plv_2, 4 * naive_coh_2, 9 * mf_first_2, plv2_2, 9 * mf_second_2]
x = np.arange(len(groups))
ax.bar(x - 0.19, vals_1, width=0.38, color=COLORS["quantum"], alpha=0.9, label="signal 1")
ax.bar(x + 0.19, vals_2, width=0.38, color=COLORS["highlight"], alpha=0.9, label="signal 2")
ax.axvline(2.5, color=COLORS["muted"], ls=":", lw=1.2)
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("value on the scale of a circular moment")
ax.set_title("Matched at the first moment, split at the second", pad=10)
ax.legend(loc="upper right")

plt.show()

**Read the figure.** The left panel shows the two engineered phase-difference distributions. They are visibly *different shapes* — signal 1 is a clean symmetric pair of spikes, signal 2 adds a central spike at $\delta = 0$ and pushes its side spikes further out — yet they were built to share the same first circular moment. Different distributions can hide behind the same first moment, and that is precisely the blind spot we are probing.

The right panel puts every reading on one scale by multiplying each entry of $\rho_{AB}$ by its prefactor. Left of the dotted line are the first-moment readings: PLV, four times the naive coherence and nine times the qutrit's first-moment entry. All three pairs of bars have the same height, near $0.83$, for both signals. Right of the line are the second-moment readings: the classical second-harmonic PLV and nine times $\lvert\rho[6,2]\rvert$. Here the bars split, and the classical bar and the quantum bar of each signal are equal. The qutrit's joint state sees the difference between the two signals, and so does the classical statistic of the same harmonic: that pairing will follow you through the whole capstone.

## 3. What the mutual information makes of it

The qutrit's mutual information was larger for signal 1 than for signal 2. Signal 2 has the *larger* second moment ($0.524$ against $0.362$). So the gap runs opposite to the second moment, and it is small, about 2% of the value. Before reading anything into it, two facts.

**The second moment has a floor.** For a symmetric difference law the moments are real, $m_1 = \langle\cos\delta\rangle$ and $m_2 = \langle\cos 2\delta\rangle$. Since $\cos 2\delta = 2\cos^2\delta - 1$ and the average of a square is at least the square of the average (the variance of $\cos\delta$ cannot be negative, a special case of Jensen's inequality),

$$ m_2 = 2\langle\cos^2\delta\rangle - 1 \;\ge\; 2\langle\cos\delta\rangle^2 - 1 = 2m_1^2 - 1 , $$

with equality exactly when $\cos\delta$ takes one value, that is, for a symmetric split $\pm s$. The symmetry only keeps the algebra real: rotating any difference law until its first moment is real gives the same bound on the moduli, $\lvert m_2\rvert \ge 2\lvert m_1\rvert^2 - 1$, for every law. Signal 1 is such a split, so it sits *on* the floor: with the first moment held at $0.825$, no difference law has a smaller second moment. Below the floor there is no law, and the joint state built from such moments is not a state at all.

**At a fixed first moment, the mutual information is not monotone in the second.** Once $\theta_A$ sweeps the circle and $\delta$ is independent of it, section 1 showed that the qutrit state depends on the moments only, so `long_run_joint_density` can build the exact long-run state from $(1, m_1, m_2)$, with no sampling noise at all. The code holds $m_1$ at signal 1's value, sweeps $m_2$ from the floor to $1$, and computes the mutual information along the way, and then repeats the sweep at two smaller first moments.

In [ ]:
m1_fixed = first_target
m2_floor = 2 * m1_fixed**2 - 1
print(f"first moment held at {m1_fixed:.4f};  floor 2 m1^2 - 1 = {m2_floor:.4f};  signal 1 has {second_1:.4f}")


def exact_qmi(m1_value, m2_value):
    rho = long_run_joint_density(np.array([1.0, m1_value, m2_value]))
    return quantum_mutual_information(rho, dims=[3, 3], base=np.e)


grid = np.linspace(m2_floor, 1.0, 401)
qmi_curve = np.array([exact_qmi(m1_fixed, v) for v in grid])
i_min = int(np.argmin(qmi_curve))
print(f"exact long-run QMI on the floor: {qmi_curve[0]:.4f} nat;  at second moment 1: {qmi_curve[-1]:.4f} nat")
print(f"exact long-run QMI is lowest at second moment {grid[i_min]:.3f}, where it is {qmi_curve[i_min]:.4f} nat")
for name, m2_value, sampled in [("signal 1", second_1, qmi_1), ("signal 2", second_2, qmi_2)]:
    print(f"{name}: second moment {m2_value:.4f}   exact QMI {exact_qmi(m1_fixed, m2_value):.4f} nat   "
          f"sampled QMI {sampled:.4f} nat")
for m2_value in [m2_floor, m2_floor - 0.1]:
    lowest = np.linalg.eigvalsh(long_run_joint_density(np.array([1.0, m1_fixed, m2_value]))).min()
    print(f"second moment {m2_value:.4f}: smallest eigenvalue of the long-run state {lowest:+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(grid, qmi_curve, color=COLORS["source"], lw=2.4, label="exact long-run QMI")
ax.axvspan(-0.1, m2_floor, color=COLORS["grid"], alpha=0.8, label="below the floor: no state")
ax.plot([second_1], [exact_qmi(m1_fixed, second_1)], "o", color=COLORS["quantum"], markersize=10, label="signal 1")
ax.plot([second_2], [exact_qmi(m1_fixed, second_2)], "o", color=COLORS["highlight"], markersize=10, label="signal 2")
ax.set_xlim(-0.1, 1.02)
ax.set_xlabel(r"second moment $m_2$  (first moment held at %.3f)" % m1_fixed)
ax.set_ylabel("quantum mutual information (nats)")
ax.set_title("Not a gauge of the second moment", pad=10)
ax.legend(loc="upper left", fontsize=9)

ax = axes[1]
for m1_value, colour in [(0.3, COLORS["flow"]), (0.6, COLORS["target"]), (m1_fixed, COLORS["source"])]:
    g = np.linspace(2 * m1_value**2 - 1, 1.0, 301)
    ax.plot(g, [exact_qmi(m1_value, v) for v in g], color=colour, lw=2.2, label=f"$m_1$ = {m1_value:.3f}")
ax.set_xlabel(r"second moment $m_2$, from the floor $2m_1^2 - 1$ to $1$")
ax.set_ylabel("quantum mutual information (nats)")
ax.set_title("The same sweep at three first moments", pad=10)
ax.legend(loc="upper left", fontsize=9)
plt.show()

**Read the output and the figure.** The floor prints $0.3624$, the second moment of signal 1 to every digit: the symmetric split sits exactly on Jensen's bound. At the floor the smallest eigenvalue of the long-run state is zero to rounding, and $0.1$ below it the eigenvalue turns negative, so the shaded band on the left really is a region with no state in it. The exact mutual information of each signal agrees with the sampled one to every printed digit, $0.3701$ and $0.3629$ nat, so the gap you saw in section 2 is a property of the two states and not of sampling noise.

*Left:* with the first moment held at $0.825$, the mutual information starts at $0.370$ nat on the floor, dips to $0.362$ nat near a second moment of $0.48$, and climbs to about $0.45$ nat at $1$. Signal 1 sits on the floor and signal 2 a little past the minimum, on the rising side but lower than the starting point, which is why the larger second moment gives the smaller value here. *Right:* the same sweep at two smaller first moments also dips and rises. There the floor is negative, and a negative second moment means the two harmonics sit half a turn apart, a relative phase the mutual information reads through its cosine. So on those curves the dip mixes the sign of the second moment with its size. The clean case is the left panel, where the floor is positive and the mutual information is not monotone even in the second-harmonic PLV.

So read that gap with care. The mutual information is not a gauge of the second moment; the entry that isolates it is the coherence $\lvert\rho[6,2]\rvert$, and the classical second-harmonic PLV reads the same number. The quantum mutual information, read alone, does not tell you which way the second moment moved.

This is not yet a measurement on a coupling problem. It is the structural precondition for a quantum measure to read anything beyond PLV, and it comes with its classical twin from the start: what the qutrit's state holds beyond PLV is the second-harmonic PLV, divided by nine.

## Your turn

**Warm-up.** Without running anything: a pair whose phase difference is an equal split between $\delta = +\pi/2$ and $\delta = -\pi/2$, independent of a $\theta_A$ that sweeps the circle. What are the PLV, the naive coherence $\lvert\rho[1,2]\rvert$ and the qutrit's $\lvert\rho[6,2]\rvert$? Which embedding can tell that the two signals are coupled at all? Check your answer with the code of section 1.

**Try it.** In the construction of section 2, comment out the line `rng.shuffle(delta)` so that each population sits on its own arc of the $\theta_A$ sweep, and rerun sections 2 and 3. Before you run, predict whether the naive mutual information of the two signals stays equal, using the rule of section 1. Then print it, together with the largest off-diagonal entry of the qutrit marginal $\rho_B$, and say which sentence of the reading after the table would have been false without the shuffle.

**Go further.** Signal 2's law is symmetric. Shift every one of its values of $\delta$ by $+0.3$ rad. Predict, by hand, the PLV, $\lvert\rho[1,2]\rvert$, $\lvert\rho[6,2]\rvert$ and the two quantum mutual informations, then check. Which readings depend on where the difference sits, and which only on how it is spread?

**Challenge.** (a) Using the rule of section 1, show that the long-run qutrit state only links basis vectors with the same $i + j$, and that its block on $\{\lvert 02\rangle, \lvert 11\rangle, \lvert 20\rangle\}$ is $\tfrac19$ times the matrix of part (b). (b) Show that the $3\times 3$ matrix $\begin{pmatrix} 1 & m_1 & m_2 \\ m_1 & 1 & m_1 \\ m_2 & m_1 & 1\end{pmatrix}$, for real $m_1, m_2$ with $m_2 \le 1$, has a non-negative determinant exactly when $m_2 \ge 2m_1^2 - 1$, and compare with the place where the long-run state of section 3 stops being a state.

## What you built

- You can show by hand that, with a phase-symmetric reference and a phase difference independent of it, the only entries of the time-averaged joint state that survive are the circular moments $\langle e^{im\delta}\rangle$ of the difference — and you have seen what leaks in when that independence is broken.
- You can locate the moments: $\lvert\rho[1,2]\rvert = \mathrm{PLV}/4$ for the phase qubit, which makes the naive embedding PLV-equivalent by construction, and $\lvert\rho[6,2]\rvert = \lvert\langle e^{2i\delta}\rangle\rvert/9$ for the qutrit, the second-harmonic PLV divided by nine.
- You can build two phase-difference signals with the same first circular moment and different second moments, and show that PLV and the naive embedding cannot separate them while the qutrit's second-moment coherence and the classical second-harmonic PLV both can.
- You can explain why the qutrit's mutual information differs between those signals without being a gauge of the second moment: at a fixed first moment it is not monotone, and for a symmetric difference law the second moment cannot fall below $2m_1^2 - 1$.

Stand back. Given any embedding of this kind, you can now list the numbers its joint state can hold, before computing a single measure — and you know the classical statistic that holds each of them too. That is the question every notebook of the capstone asks first.

## What's next

Today's two signals were built by hand, with their moments set to order. In `04/12_kuramoto_dyad_and_ground_truth` you build a synthetic pair of coupled oscillators with a *known* coupling strength, the ground truth against which every measure is judged, together with the classical baselines that already read it. The capstone (`04/12`–`04/20`) then runs the experiment that today's toy only prepares: it sets a classical statistic beside every quantum measure and writes down what each one reads. You will find that a quantum coupling measure whose embedding carries the second harmonic reads past PLV as far as a classical statistic of the same two harmonics and no further, less far on their relative phase, and what quantum optimal transport brings is one common geometry in which to ask the question.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments of a phase distribution; the first and second moments are distinct functionals.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information $I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$.

**Previous:** `notebooks/04_QuantumOptimalTransport/10_embedding_signals_into_states.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/12_kuramoto_dyad_and_ground_truth.ipynb`